# 🏥 EHDS AMR Data Validation Demo

End-to-end demonstration of federated data validation using the **European Health Data Space (EHDS) Antimicrobial Resistance** Data Product.

## Workflow Overview

| Phase | Step | Description |
|-------|------|-------------|
| **1** | Federation Population | Register and federate the EHDS AMR Data Product into the Semantic Data Model |
| **2** | Data Validation | Plan, execute, and inspect policy-driven validations (DQR1EH–DQR6EH) |

**Standards**: DCAT · CSVW · ODRL · Schema.org · PROV-O · ISO 3166

---

## 🔧 Setup

In [ ]:
import os
import sys
import subprocess
import json
import papermill as pm
from pathlib import Path
from rdflib import Graph, Namespace, RDF, RDFS, Literal, URIRef

In [ ]:
# Project Root (one level up from demo/)
base_dir = str(Path(os.getcwd()).parent) if os.path.basename(os.getcwd()) == 'demo' else os.getcwd()

# Key Paths
PATHS = {
    'sdm':        os.path.join(base_dir, 'FederatedComputationalGovernance/SemanticDataModel/sdm.ttl'),
    'tbox':       os.path.join(base_dir, 'FederatedComputationalGovernance/FederatedTeam/tbox_standards.ttl'),
    'cdm':        os.path.join(base_dir, 'FederatedComputationalGovernance/GlobalDefinitions/common_data_models.json'),
    'populator':  os.path.join(base_dir, 'FederatedComputationalGovernance/federated_layer_populator.ipynb'),
    'profiler':   os.path.join(base_dir, 'DataPlatformLayer/Registration/profiler.ipynb'),
    'federator':  os.path.join(base_dir, 'DataPlatformLayer/Integration/federator.ipynb'),
    'dp1_json':   os.path.join(base_dir, 'DataPlatformLayer/Integration/dp1.json'),
    'data_csv':   os.path.join(base_dir, 'DataProductLayer/DataProduct_EHDS_AMR/Data/Patient_Summary.csv'),
    'planner_py': os.path.join(base_dir, 'Connector/ValidationFramework/planner/planner_n3_semantic.py'),
    'planner_dir':os.path.join(base_dir, 'Connector/ValidationFramework/planner/'),
    'executor_py':os.path.join(base_dir, 'Connector/ValidationFramework/executor/executor_semantic.py'),
    'executor_dir':os.path.join(base_dir, 'Connector/ValidationFramework/executor/'),
    'code_meta':  os.path.join(base_dir, 'Connector/ValidationFramework/executor/code_metadata_with_roles.json'),
}

# Namespaces
tbox = Namespace('http://www.semanticweb.org/acraf/ontologies/2024/healthmesh/tbox#')
abox = Namespace('http://www.semanticweb.org/acraf/ontologies/2024/healthmesh/abox#')

print(f"✓ Base directory: {base_dir}")
print(f"✓ Data file: {os.path.basename(PATHS['data_csv'])}")
print(f"✓ Integration contract: {os.path.basename(PATHS['dp1_json'])}")

<a id="phase1"></a>
# 📋 PHASE 1: Federation Population

Register the EHDS AMR Patient Summary dataset into the federated Semantic Data Model.

| Step | Notebook | Purpose |
|------|----------|---------|
| 1.1 | `federated_layer_populator.ipynb` | Initialize SDM with TBox + CDM + ODRL policies |
| 1.2 | `profiler.ipynb` | Extract metadata from `Patient_Summary.csv` |
| 1.3 | `federator.ipynb` | Create data contract with schema mappings + policy bindings |

---

## Step 1.1: Initialize Semantic Data Model

Build the SDM foundation from the TBox ontology, Common Data Model (`EHDS_Patient_Summary`), and ODRL policies (DQRP1–DQRP7).

In [ ]:
pm.execute_notebook(
    PATHS['populator'],
    None,
    parameters={
        'folder': os.path.join(base_dir, 'FederatedComputationalGovernance/'),
    },
    kernel_name='python3',
    log_output=False
)
print("✓ SDM initialized with TBox + CDM + ODRL policies")

## Step 1.2: Register EHDS AMR Data Product

The Profiler extracts metadata from `Patient_Summary.csv` and creates a DCAT-compliant DataProduct in the SDM.

Columns profiled: `nationalHealthcarePatientID`, `familyName`, `givenName`, `dateOfBirth`, `gender`, `countryOfAffiliation`, `hospitalCode`, `hospitalCountry`, `lastUpdated`

In [ ]:
file_path = PATHS['data_csv']
print(f"Registering: {file_path}")

In [ ]:
pm.execute_notebook(
    PATHS['profiler'],
    None,
    parameters={
        'folder': os.path.join(base_dir, 'DataPlatformLayer/Registration'),
        'file_path': file_path
    },
    kernel_name='python3',
    log_output=False
)
print("✓ EHDS AMR Patient Summary registered in SDM")

## Step 1.3: Federate with Schema Mappings & Policy Binding

The Federator creates a **Data Contract** linking the EHDS data product to:
- **Schema mappings**: Physical columns → semantic features (e.g., `gender` → `Sex`)
- **Policy bindings**: DQRP1 (Timeliness) attached to the data contract

In [ ]:
# Inspect the integration contract
with open(PATHS['dp1_json']) as f:
    contract = json.load(f)
print(json.dumps(contract, indent=2))

In [ ]:
pm.execute_notebook(
    PATHS['federator'],
    None,
    parameters={
        'folder': os.path.join(base_dir, 'DataPlatformLayer/Integration'),
        'dp_meta_path': PATHS['dp1_json'],
    },
    kernel_name='python3',
    log_output=False
)
print("✓ Data contract created with schema mappings + DQRP1 policy")

## Step 1.4: Inspect the Federated SDM

Query the Semantic Data Model to verify the registered data product, its attributes, and policy bindings.

In [ ]:
sdm = Graph().parse(PATHS['sdm'], format='turtle')
print(f"✓ SDM loaded: {len(sdm):,} triples")

In [ ]:
# Query: Registered data products
query = '''
PREFIX tb: <http://www.semanticweb.org/acraf/ontologies/2024/healthmesh/tbox#>
PREFIX dcterms: <http://purl.org/dc/terms/>

SELECT ?dp ?title ?format
WHERE {
    ?dp a tb:DataProduct .
    OPTIONAL { ?dp dcterms:title ?title }
    OPTIONAL { ?dp tb:hasDTT ?format }
}
'''
print("📊 Registered Data Products:")
print("-" * 60)
for row in sdm.query(query):
    name = str(row.dp).split('#')[-1]
    title = str(row.title) if row.title else 'N/A'
    fmt = str(row.format).split('#')[-1] if row.format else 'N/A'
    print(f"  {name:40s} [{fmt}]")

In [ ]:
# Query: Data product attributes (columns)
query = '''
PREFIX tb: <http://www.semanticweb.org/acraf/ontologies/2024/healthmesh/tbox#>
PREFIX csvw: <http://www.w3.org/ns/csvw#>

SELECT ?dp ?attr
WHERE {
    ?dp a tb:DataProduct ;
        csvw:column ?attr .
}
ORDER BY ?dp ?attr
'''
print("📋 Data Product Attributes:")
print("-" * 60)
for row in sdm.query(query):
    dp_name = str(row.dp).split('#')[-1]
    attr_name = str(row.attr).split('#')[-1]
    print(f"  {dp_name:40s} → {attr_name}")

In [ ]:
# Query: Policy bindings on data contracts
query = '''
PREFIX tb: <http://www.semanticweb.org/acraf/ontologies/2024/healthmesh/tbox#>

SELECT ?dp ?contract ?policy
WHERE {
    ?dp a tb:DataProduct ;
        tb:hasDC ?contract .
    ?contract tb:hasPolicy ?policy .
}
'''
print("🔒 Policy Bindings:")
print("-" * 60)
for row in sdm.query(query):
    dp_name = str(row.dp).split('#')[-1]
    policy_name = str(row.policy).split('#')[-1]
    print(f"  {dp_name:40s} ← {policy_name}")

<a id="phase2"></a>
# ✅ PHASE 2: Data Validation Workflow

Validate the EHDS AMR data product against governance policies.

| Step | Component | Description |
|------|-----------|-------------|
| 2.1 | Query CDMs | Discover available Common Data Models |
| 2.2 | Planner | Generate PolicyCheckers from N3 reasoning rules |
| 2.3 | Executor | Execute validation chains and store reports |
| 2.4 | Analysis | Query validation results via SPARQL |

---

## Step 2.1: Query Available Common Data Models

In [ ]:
query = '''
PREFIX tb: <http://www.semanticweb.org/acraf/ontologies/2024/healthmesh/tbox#>

SELECT ?cdm
WHERE {
    ?cdm a tb:CommonDataModel .
}
'''

cdms = [row.cdm.split('#')[1] for row in sdm.query(query)]
print(f"Available Common Data Models: {cdms}")

# Select the EHDS CDM
dp = "EHDS_Patient_Summary"
print(f"\n→ Selected CDM: {dp}")

## Step 2.2: Retrieve Data Products with Policies

In [ ]:
query = '''
PREFIX tb: <http://www.semanticweb.org/acraf/ontologies/2024/healthmesh/tbox#>

SELECT DISTINCT ?dataset
WHERE {
    ?dataset a tb:DataProduct .
}
'''

qres = sdm.query(query)
data_products = set(row.dataset.split('#')[1] for row in qres)

print(f"Found {len(data_products)} data product(s):")
for dp_name in sorted(data_products):
    print(f"  - {dp_name}")

## Step 2.3: Generate Policy Checkers (Planner)

The planner uses **N3 reasoning rules** to generate PolicyCheckers for each (DataProduct, Policy) pair.

Each PolicyChecker contains an operation chain: `LoadData → Check → Constraint → Report`

In [ ]:
for dp_name in sorted(data_products):
    print(f"📋 Generating PolicyCheckers for: {dp_name}")
    
    result = subprocess.run(
        ['python', PATHS['planner_py'], PATHS['sdm'], dp_name],
        cwd=PATHS['planner_dir'],
        capture_output=True,
        text=True
    )
    
    if result.returncode == 0:
        print(f"   ✓ Success: {result.stdout.strip()}")
    else:
        print(f"   ✗ Error: {result.stderr[:500]}")
        raise Exception(f"Planner failed for {dp_name}")

print("\n" + "=" * 60)
print("✅ All PolicyCheckers generated and merged into SDM")
print("=" * 60)

## Step 2.4: Execute Validations (Executor)

The executor:
1. Reads PolicyCheckers from SDM
2. Translates operation chains into composed UDFs
3. Binds parameters (file paths, thresholds, vocabularies)
4. Executes validations and stores **Validation Reports** in SDM

In [ ]:
# Reload SDM after planner has merged PolicyCheckers
sdm = Graph().parse(PATHS['sdm'], format='turtle')
print(f"✓ Reloaded SDM with PolicyCheckers: {len(sdm):,} triples")
print()

# Find all PolicyCheckers for the EHDS data product
dp_uri = abox['Patient_Summary']
policy_checkers = list(sdm.subjects(tbox.validates, dp_uri))

print(f"Found {len(policy_checkers)} PolicyChecker(s) for Patient_Summary")
print()

# Execute each PolicyChecker
for pc_uri in policy_checkers:
    policy = sdm.value(pc_uri, tbox.accordingTo)
    policy_name = str(policy).split('#')[-1] if policy else 'Unknown'
    
    print(f"⚙️  Executing PolicyChecker for policy: {policy_name}")
    print(f"   URI: {pc_uri}")
    
    result = subprocess.run(
        ['python', PATHS['executor_py'], str(pc_uri), PATHS['code_meta']],
        cwd=PATHS['executor_dir'],
        capture_output=True,
        text=True
    )
    
    if result.returncode == 0:
        print(f"   ✓ Success")
    else:
        print(f"   ✗ Error: {result.stderr[:500]}")
    print()

print("=" * 60)
print("✅ Validation execution complete!")
print("=" * 60)

<a id="verification"></a>
# 📊 Verification & Analysis

Query the SDM to retrieve validation results and analyze governance compliance.

---

In [ ]:
# Reload SDM with validation reports
sdm = Graph().parse(PATHS['sdm'], format='turtle')
print(f"✓ Loaded SDM (single source of truth): {len(sdm):,} triples")

## Query 1: Validation Results Summary

Retrieve validation status for each (DataProduct, Policy) pair:

In [ ]:
query = '''
PREFIX tb: <http://www.semanticweb.org/acraf/ontologies/2024/healthmesh/tbox#>
PREFIX prov: <http://www.w3.org/ns/prov#>

SELECT ?dataset ?policy ?status ?result ?duration
WHERE {
    ?pc a tb:PolicyChecker ;
        tb:validates ?dataset ;
        tb:accordingTo ?policy ;
        tb:hasValidationReport ?report .
    ?report tb:validationStatus ?status .
    OPTIONAL { ?report tb:resultValue ?result }
    OPTIONAL { ?report tb:executionDuration ?duration }
}
ORDER BY ?dataset ?policy
'''

print(f"{'Dataset':<35} {'Policy':<12} {'Status':<12} {'Result':<12} {'Duration (s)'}")
print("=" * 85)
for row in sdm.query(query):
    ds = str(row.dataset).split('#')[-1]
    pol = str(row.policy).split('#')[-1]
    status = str(row.status)
    res = str(row.result) if row.result else 'N/A'
    dur = f"{float(row.duration):.4f}" if row.duration else 'N/A'
    emoji = '✅' if status == 'PASSED' else '❌' if status == 'FAILED' else '⚠️'
    print(f"  {emoji} {ds:<33} {pol:<12} {status:<12} {res:<12} {dur}")

## Query 2: Detailed Validation Reports

In [ ]:
query = '''
PREFIX tb: <http://www.semanticweb.org/acraf/ontologies/2024/healthmesh/tbox#>
PREFIX prov: <http://www.w3.org/ns/prov#>

SELECT ?report ?status ?resultType ?resultValue ?mode ?start ?end
WHERE {
    ?report a tb:ValidationReport ;
        tb:validationStatus ?status .
    OPTIONAL { ?report tb:resultType ?resultType }
    OPTIONAL { ?report tb:resultValue ?resultValue }
    OPTIONAL { ?report tb:executionMode ?mode }
    OPTIONAL { ?report prov:startedAtTime ?start }
    OPTIONAL { ?report prov:endedAtTime ?end }
}
ORDER BY ?start
'''

print("📋 Detailed Validation Reports:")
print("=" * 80)
for row in sdm.query(query):
    report_id = str(row.report).split('#')[-1]
    status = str(row.status)
    emoji = '✅' if status == 'PASSED' else '❌' if status == 'FAILED' else '⚠️'
    print(f"\n  {emoji} {report_id}")
    print(f"     Status:  {status}")
    if row.resultType:
        print(f"     Type:    {row.resultType}")
    if row.resultValue:
        print(f"     Value:   {row.resultValue}")
    if row.mode:
        print(f"     Mode:    {row.mode}")
    if row.start:
        print(f"     Time:    {row.start} → {row.end}")

## Summary

This demo demonstrated the complete lifecycle of the EHDS AMR Data Product through:

1. **Registration** — Profiled `Patient_Summary.csv` into DCAT/CSVW metadata
2. **Integration** — Created a data contract mapping physical columns to EHDS semantic features
3. **Validation** — Generated and executed PolicyCheckers for DQR1EH–DQR6EH

All metadata, contracts, and validation results are stored in the **Semantic Data Model** (SDM) as a single source of truth, queryable via SPARQL.

---
*FEED Research Group · UPC · 2026*